In [4]:
%pip install torch
%pip install transformers
%pip install accelerate
%pip install bitsandbytes
%pip install hf_transfer

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from repeng import ControlVector, ControlModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

# (If needed on your laptop) login to HF first:
# huggingface-cli login

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0

# # Use 4-bit so it fits on laptop GPUs
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb,
#     device_map="auto",
# )

# Load model in float16 on Apple MPS if available, otherwise CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "mps" else torch.float32,
    device_map=None  # no accelerate needed
).to(device)

# Wrap with ControlModel (use the same layer ids you trained with)
model = ControlModel(base_model, list(range(-5, -18, -1)))

# Load the trained control vector
cv = ControlVector.import_gguf("data/control_vector.gguf")

# Apply it (tune coeff as you like)
model.set_control(cv, coeff=1.0)

# Now generate with the steered model
inp = tokenizer("[INST] Write a cheerful greeting [/INST]", return_tensors="pt").to(model.device)
out = model.generate(**inp, max_new_tokens=80)
print(tokenizer.decode(out[0], skip_special_tokens=True))


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:42<00:00, 21.46s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
